# SpectraLite

## Phase 0 — Environment Setup & Verification

This notebook validates the SpectraLite research environment. It does **not** implement compression, SVD, whitening, rank allocation, or latency engineering.

**Objectives**
1. Install Phase 0 dependencies
2. Verify Python / PyTorch / CUDA / GPU visibility
3. Load `facebook/opt-125m` (FP16 on CUDA, `device_map="auto"`)
4. Inventory every `nn.Linear` layer for later SVD targeting
5. Run one short generation smoke test
6. Report GPU memory before load / after load / after inference

All logic lives in the `spectralite` package; cells below only orchestrate the experiment.

### Install Dependencies

Installs the pinned Phase 0 stack from `requirements.txt`. Safe to re-run; pip will skip satisfied packages.

> **Colab tip:** Runtime → Restart session after the first install if imports look stale, then continue from *Environment Verification*.

In [3]:
import subprocess
import sys
from pathlib import Path

# Resolve repository root whether the notebook runs from SpectraLite/
# or from SpectraLite/notebooks/ (Colab / local).
cwd = Path.cwd().resolve()
candidates = [cwd, cwd.parent]
repo_root = None
for candidate in candidates:
    if (candidate / "spectralite").is_dir() and (candidate / "requirements.txt").is_file():
        repo_root = candidate
        break

if repo_root is None:
    raise FileNotFoundError(
        "Could not locate SpectraLite repo root (expected spectralite/ + requirements.txt)."
    )

requirements = repo_root / "requirements.txt"
print(f"Repo root      : {repo_root}")
print(f"Requirements   : {requirements}")

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements)]
)
print("Dependencies installed.")

Repo root      : /content/Execution/SpectraLite
Requirements   : /content/Execution/SpectraLite/requirements.txt
Dependencies installed.


In [2]:
!git clone https://github.com/PrabinDevkota/Execution.git
import sys
sys.path.insert(0, "/content/Execution/SpectraLite")
%cd /content/Execution/SpectraLite/notebooks
!ls ../../spectralite

Cloning into 'Execution'...
remote: Enumerating objects: 18, done.
remote: Counting objects: 100% (18/18), done.
remote: Compressing objects: 100% (14/14), done.
remote: Total 18 (delta 0), reused 18 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (18/18), 18.50 KiB | 2.31 MiB/s, done.
/content/Execution/SpectraLite/notebooks
ls: cannot access '../../spectralite': No such file or directory


### Environment Verification

Adds the repo to `sys.path`, seeds RNGs, ensures output directories exist, and prints the Phase 0 environment report (Python, PyTorch, CUDA, GPU).

In [4]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
repo_root = None
for candidate in [cwd, cwd.parent]:
    if (candidate / "spectralite").is_dir():
        repo_root = candidate
        break
assert repo_root is not None, "SpectraLite repo root not found"

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from spectralite import __version__, default_config, set_seed
from spectralite.system import print_environment_report
from spectralite.utils import get_logger, print_section

cfg = default_config()
cfg.ensure_directories()
set_seed(cfg.seed)
logger = get_logger("spectralite.phase0", level=cfg.log_level)

print_section("SpectraLite Phase 0")
print(f"  Package version : {__version__}")
print(f"  Model           : {cfg.model_name}")
print(f"  Seed            : {cfg.seed}")
print(f"  Dtype preference: {cfg.dtype}")
print(f"  Device map      : {cfg.device_map}")

env_info = print_environment_report()
environment_ready = env_info["pytorch_version"] != "unavailable"
gpu_ready = bool(env_info["cuda_available"])
logger.info("Environment ready=%s | GPU ready=%s", environment_ready, gpu_ready)


 SpectraLite Phase 0
  Package version : 0.1.0-phase0
  Model           : facebook/opt-125m
  Seed            : 42
  Dtype preference: float16
  Device map      : auto

 Environment Verification
  Python Version               3.12.13
  PyTorch Version              2.11.0+cpu
  CUDA Version                 n/a (CPU build or CUDA unavailable)
  cuDNN Version                n/a
  GPU Name                     CPU (CUDA unavailable)
  GPU Memory                   n/a
  Torch CUDA Available         False
  Device                       cpu
  GPU Count                    0
  Platform                     Linux-6.6.122+-x86_64-with-glibc2.35
2026-07-12 14:23:56 | INFO     | spectralite.phase0 | Environment ready=True | GPU ready=False


### GPU Information

Captures a baseline GPU memory snapshot **before** the model is loaded. On CPU-only machines this section reports that CUDA is unavailable and continues.

In [5]:
from spectralite import gpu

mem_before = gpu.snapshot(label="Memory before loading")
gpu.print_memory_snapshot(mem_before)

print("\n  CUDA available :", gpu.is_cuda_available())
if gpu.is_cuda_available():
    print("  Total VRAM     :", mem_before["total"])
    print("  Free (approx)  :", mem_before["free"])


 Memory before loading
  Status                       CUDA unavailable — CPU mode

  CUDA available : False


### Load Model

Loads `facebook/opt-125m` via `AutoTokenizer` + `AutoModelForCausalLM`.

- **Dtype:** FP16 when CUDA is available, otherwise FP32
- **Placement:** `device_map="auto"`
- **Mode:** `eval()` (no training in Phase 0)

In [6]:
from spectralite.model_loader import (
    load_model_and_tokenizer,
    print_model_load_summary,
)

model, tokenizer = load_model_and_tokenizer(config=cfg)
load_summary = print_model_load_summary(model, model_name=cfg.model_name)

mem_after_load = gpu.snapshot(label="Memory after loading")
gpu.print_memory_snapshot(mem_after_load)

model_loaded = model is not None and tokenizer is not None
logger.info("Model loaded: %s", cfg.model_name)

2026-07-12 14:24:16 | INFO     | spectralite.model_loader | Loading tokenizer: facebook/opt-125m


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/685 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/251M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]


 Model Load Summary
  Model Name                   facebook/opt-125m
  Architecture                 OPTForCausalLM
  Total Parameters             125.24M (125,239,296)
  Trainable Parameters         125.24M (125,239,296)
  Model Size (params)          238.88 MB
  Tensor Data Type             float16
  Current Device               cpu

 Memory after loading
  Status                       CUDA unavailable — CPU mode
2026-07-12 14:24:23 | INFO     | spectralite.phase0 | Model loaded: facebook/opt-125m


### Model Analysis

Prints architecture statistics and **every** `nn.Linear` layer (name, in/out features, weight shape). This inventory is the targeting surface for later SVD compression phases — Phase 0 only inspects.

In [7]:
from spectralite.model_analysis import print_full_model_analysis

analysis = print_full_model_analysis(model, model_name=cfg.model_name)

print(f"\n  Block names (first 3): {analysis['block_names'][:3]}")
print(f"  Linear layers catalogued: {analysis['num_linear_layers']}")


 Model Information
  Model Name                   facebook/opt-125m
  Architecture                 OPTForCausalLM
  Transformer Blocks           12
  Attention Linear Layers      48
  MLP Linear Layers            24
  Other Linear Layers          1
  Total nn.Linear Layers       73
  Total Parameters             125.24M (125,239,296)
  Trainable Parameters         125.24M (125,239,296)
  Model Size / Memory Footprint 238.88 MB
  Tensor Data Type             float16
  Current Device               cpu

 nn.Linear Inventory (73 layers)
  #    Layer Name                                                  In    Out Weight Shape    
  -------------------------------------------------------------------------------------------
  0    model.decoder.layers.0.self_attn.k_proj                    768    768 (768, 768)      
  1    model.decoder.layers.0.self_attn.v_proj                    768    768 (768, 768)      
  2    model.decoder.layers.0.self_attn.q_proj                    768    768 (768, 7

### Test Inference

Single smoke-test generation to confirm the forward / generate path works end-to-end.

- **Prompt:** *Explain Singular Value Decomposition in one sentence.*
- **Budget:** ~50 new tokens (greedy decoding for reproducibility)

In [8]:
from spectralite.model_loader import generate_text
from spectralite.utils import print_section

prompt = cfg.smoke_prompt
generated = generate_text(
    model,
    tokenizer,
    prompt,
    max_new_tokens=cfg.max_new_tokens,
    do_sample=False,
)

print_section("Test Inference")
print("  Prompt:")
print(f"    {prompt}")
print("\n  Model output:")
print("  " + "-" * 60)
for line in generated.strip().splitlines() or [generated.strip()]:
    print(f"  {line}")
print("  " + "-" * 60)

inference_ok = isinstance(generated, str) and len(generated.strip()) > 0
logger.info("Inference successful=%s | chars=%d", inference_ok, len(generated))


 Test Inference
  Prompt:
    Explain Singular Value Decomposition in one sentence.

  Model output:
  ------------------------------------------------------------
  Explain Singular Value Decomposition in one sentence.
  
  I’m not sure what you mean by “explain”.
  
  I’m not sure what you mean by “explain”.
  
  I’m not sure what you mean by
  ------------------------------------------------------------
2026-07-12 14:24:42 | INFO     | spectralite.phase0 | Inference successful=True | chars=168


### GPU Memory

Final memory snapshot after inference, then a three-stage timeline (before load → after load → after inference).

In [9]:
mem_after_infer = gpu.snapshot(label="Memory after inference")
gpu.print_memory_snapshot(mem_after_infer)

gpu.print_memory_timeline([mem_before, mem_after_load, mem_after_infer])


 Memory after inference
  Status                       CUDA unavailable — CPU mode

 GPU Memory Timeline
  Status                       CUDA unavailable — CPU mode
  Memory before loading        n/a
  Memory after loading         n/a
  Memory after inference       n/a


### Summary

Phase 0 completion checklist. All items should read **PASS** before moving to Phase 1.

In [10]:
from spectralite.utils import print_checklist, print_section

phase0_complete = all([environment_ready, model_loaded, inference_ok])

print_checklist(
    [
        ("Environment Ready", environment_ready),
        ("GPU Ready", gpu_ready),
        ("Model Loaded Successfully", model_loaded),
        ("Inference Successful", inference_ok),
        ("Phase 0 Complete", phase0_complete),
    ]
)

print_section("Phase 0 Outcome")
print("  Environment Ready" if environment_ready else "  Environment NOT Ready")
print("  GPU Ready" if gpu_ready else "  GPU NOT Ready (CPU fallback OK for Phase 0 smoke test)")
print("  Model Loaded Successfully" if model_loaded else "  Model Load FAILED")
print("  Inference Successful" if inference_ok else "  Inference FAILED")
print("  Phase 0 Complete" if phase0_complete else "  Phase 0 INCOMPLETE")


 Phase 0 Status
  [PASS] Environment Ready
  [FAIL] GPU Ready
  [PASS] Model Loaded Successfully
  [PASS] Inference Successful
  [PASS] Phase 0 Complete

 Phase 0 Outcome
  Environment Ready
  GPU NOT Ready (CPU fallback OK for Phase 0 smoke test)
  Model Loaded Successfully
  Inference Successful
  Phase 0 Complete
